# KCORC Summer School - TESPy Workshop

## Case Study: 5.5 MWe Double-stage ORC Kirchstockach

The Kirchstockach geothermal power plant is a two-stage Organic Rankine Cycle
(ORC) system consisting of a High-Temperature (HT) ORC and a Low-Temperature
(LT) ORC operating in series. The geothermal brine first transfers heat to the
HT cycle and subsequently to the LT cycle before reinjection.

![ORC flowsheet](../orc.svg)

Figure 1: Scheme of the double-stage ORC power plant in Kirchstockach, Germany
(Florian Heberle, Thomas Jahrfeld and Dieter Brüggemann, 2015)
https://worldgeothermal.org/pdf/IGAstandard/WGC/2015/26002.pdf


## Validated model

This notebook contains a validated set of boundary conditions for the design
point model of the full system. You can use it as a starting point for your
tasks.

In [ ]:
from tespy.networks import Network
from tespy.components import (
    Source, Sink,
    Pump, Turbine,
    SectionedHeatExchanger,
    CycleCloser,
    Splitter, Merge,
    PowerSink, PowerBus,
    Motor, Generator
)
from tespy.connections import Connection, PowerConnection

import numpy as np
import matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI

In [ ]:
nw = Network()

nw.units.set_defaults(
    temperature="degC", pressure="bar", pressure_difference="bar",
    enthalpy="kJ/kg", mass_flow="kg/s", power="kW", heat="kW"
)

In [ ]:
ht_cc = CycleCloser("HT-cycle-closer")

ht_pump = Pump("HT-pump")
hht_preheater = SectionedHeatExchanger("HHT-preheater")
lht_preheater = SectionedHeatExchanger("LHT-preheater")
ht_evaporator = SectionedHeatExchanger("HT-evaporator")
ht_turbine = Turbine("HT-turbine")
ht_condenser = SectionedHeatExchanger("HT-condenser")

ht_air_source = Source("HT-air-source")
ht_air_sink = Sink("HT-air-sink")

lt_cc = CycleCloser("LT-cycle-closer")

lt_pump = Pump("LT-pump")
lt_preheater = SectionedHeatExchanger("LT-preheater")
lt_evaporator = SectionedHeatExchanger("LT-evaporator")
lt_turbine = Turbine("LT-turbine")
lt_condenser = SectionedHeatExchanger("LT-condenser")

lt_air_source = Source("LT-air-source")
lt_air_sink = Sink("LT-air-sink")

geo_source = Source("geothermal-source")
geo_split = Splitter("geothermal-splitter", num_out=2)
geo_merge = Merge("geothermal-merge", num_in=2)
geo_sink = Sink("geothermal-sink")

ht_generator = Generator("HT-generator")
ht_motor = Motor("HT-motor")

lt_generator = Generator("LT-generator")
lt_motor = Motor("LT-motor")

distribution = PowerBus("power-distribution", num_in=2, num_out=3)
grid = PowerSink("grid")

In [ ]:
# HT

c1 = Connection(ht_condenser, "out1", ht_pump, "in1", label="01")
c2 = Connection(ht_pump, "out1", lht_preheater, "in2", label="02")
c3 = Connection(lht_preheater, "out2", hht_preheater, "in2", label="03")
c4 = Connection(hht_preheater, "out2", ht_evaporator, "in2", label="04")
c5 = Connection(ht_evaporator, "out2", ht_turbine, "in1", label="05")
c5a = Connection(ht_turbine, "out1", ht_cc, "in1", label="05a")
c6 = Connection(ht_cc, "out1", ht_condenser, "in1", label="06")

nw.add_conns(c1, c2, c3, c4, c5, c5a, c6)

ht_a1 = Connection(ht_air_source, "out1", ht_condenser, "in2", label="HT-a1")
ht_a2 = Connection(ht_condenser, "out2", ht_air_sink, "in1", label="HT-a2")

nw.add_conns(ht_a1, ht_a2)

# LT

c7 = Connection(lt_condenser, "out1", lt_pump, "in1", label="07")
c8 = Connection(lt_pump, "out1", lt_preheater, "in2", label="08")
c9 = Connection(lt_preheater, "out2", lt_evaporator, "in2", label="09")
c10 = Connection(lt_evaporator, "out2", lt_turbine, "in1", label="10")
c10a = Connection(lt_turbine, "out1", lt_cc, "in1", label="10a")
c11 = Connection(lt_cc, "out1", lt_condenser, "in1", label="11")

nw.add_conns(c7, c8, c9, c10, c10a, c11)

lt_a1 = Connection(lt_air_source, "out1", lt_condenser, "in2", label="LT-a1")
lt_a2 = Connection(lt_condenser, "out2", lt_air_sink, "in1", label="LT-a2")

nw.add_conns(lt_a1, lt_a2)

# Geothermal

gA = Connection(geo_source, "out1", ht_evaporator, "in1", label="A")
gB = Connection(ht_evaporator, "out1", hht_preheater, "in1", label="B")
gC = Connection(hht_preheater, "out1", lt_evaporator, "in1", label="C")
gD1 = Connection(lt_evaporator, "out1", geo_split, "in1", label="D1")
gE = Connection(geo_split, "out1", lt_preheater, "in1", label="E")
gD2= Connection(geo_split, "out2", lht_preheater, "in1", label="D2")
gF= Connection(lt_preheater, "out1", geo_merge, "in1", label="F")
gG= Connection(lht_preheater, "out1", geo_merge, "in2", label="G")
gH= Connection(geo_merge, "out1", geo_sink, "in1", label="H")

nw.add_conns(gA, gB, gC, gD1, gD2, gE, gF, gG, gH)

# Power

ht_e1 = PowerConnection(ht_turbine, "power", ht_generator, "power_in", label="HT-e1")
ht_e2 = PowerConnection(ht_generator, "power_out", distribution, "power_in1", label="HT-e2")

ht_e3 = PowerConnection(distribution, "power_out1", ht_motor, "power_in", label="HT-e3")
ht_e4 = PowerConnection(ht_motor, "power_out", ht_pump, "power", label="HT-e4")

lt_e1 = PowerConnection(lt_turbine, "power", lt_generator, "power_in", label="LT-e1")
lt_e2 = PowerConnection(lt_generator, "power_out", distribution, "power_in2", label="LT-e2")

lt_e3 = PowerConnection(distribution, "power_out2", lt_motor, "power_in", label="LT-e3")
lt_e4 = PowerConnection(lt_motor, "power_out", lt_pump, "power", label="LT-e4")


e5 = PowerConnection(distribution, "power_out3", grid, "power", label="e5")

nw.add_conns(ht_e1, ht_e2, ht_e3, ht_e4, lt_e1, lt_e2, lt_e3, lt_e4, e5)

In [ ]:
c1.set_attr(fluid={"R245fa": 1}, p=1.49, td_bubble=2)
c4.set_attr(td_bubble=2.5)
c5.set_attr(p=13.85, x=1)

ht_a1.set_attr(fluid={"air": 1}, p=1, T=8.67)
ht_a2.set_attr(T=19)

c7.set_attr(fluid={"R245fa": 1}, p=1.55, td_bubble=3)
c8.set_attr(p=6.75)
c9.set_attr(td_bubble=2)
c10.set_attr(x=1)

gA.set_attr(fluid={"water": 1}, T=135.1, p=10, m=122.4)
gC.set_attr(T=92.95)
gD1.set_attr(T=70.4)
gD2.set_attr(m=67.9)
gG.set_attr(T=51.66)

lt_a1.set_attr(fluid={"air": 1}, p=1, T=8.67)
lt_a2.set_attr(T=19)

In [ ]:
ht_turbine.set_attr(eta_s=0.883)
ht_pump.set_attr(eta_s=0.52)   # paper: isentropic efficiency calculated from design data (52 %)
ht_generator.set_attr(eta=1)
ht_motor.set_attr(eta=1)

ht_condenser.set_attr(dp1=0, dp2=0)
lht_preheater.set_attr(dp1=0, dp2=1.39 / 3)
hht_preheater.set_attr(dp1=0, dp2=1.39 / 3)
ht_evaporator.set_attr(dp1=0, dp2=1.39 / 3)

lt_evaporator.set_attr(dp1=0, dp2=0)
lt_preheater.set_attr(dp2=0.78)  # no dp1 as is parallel to lht path!
lt_condenser.set_attr(dp1=0, dp2=0)

lt_turbine.set_attr(eta_s=0.827)
lt_pump.set_attr(eta_s=0.50)   # paper: isentropic efficiency calculated from design data (50 %)
lt_generator.set_attr(eta=1)
lt_motor.set_attr(eta=1)

In [ ]:
nw.solve("design")

## Results

Key performance indicators (power, efficiencies, brine exergy flow, total UA)
are calculated with plain helper functions operating on the `Network` instance
and its connections.

In [ ]:
# helper functions for calculated properties (replacing the ModelTemplate lookup)
T0 = 8.67 + 273.15  # ambient reference temperature (paper design point)
p0 = 1.013e5


def exergy_flow_geo():
    # exergy flow of the brine at production state A, Eq. (4)/(5) of the paper [W]
    h0 = PropsSI("H", "T", T0, "P", p0, "water")
    s0 = PropsSI("S", "T", T0, "P", p0, "water")
    sA = PropsSI("S", "P", gA.p.val_SI, "H", gA.h.val_SI, "water")
    return gA.m.val_SI * ((gA.h.val_SI - h0) - T0 * (sA - s0))


def gross_power():
    # turbine shaft power HT + LT [kW] (generator inputs)
    return (ht_e1.E.val_SI + lt_e1.E.val_SI) / 1000


def net_power():
    # electrical power delivered to the grid [kW]
    return e5.E.val


def heat_input():
    # heat extracted from the brine between production (A) and reinjection (H) [kW]
    return gA.m.val_SI * (gA.h.val_SI - gH.h.val_SI) / 1000


def thermal_efficiency():
    return net_power() / heat_input()


def eta_II_gross():
    # gross second law efficiency, Eq. (2) of the paper [%]
    return 100 * gross_power() * 1000 / exergy_flow_geo()


def eta_II_net():
    # net second law efficiency, Eq. (3) of the paper [%]
    return 100 * e5.E.val_SI / exergy_flow_geo()


def plant_UA():
    # sum of the UA values of all heat exchangers [W/K]
    return nw.results["SectionedHeatExchanger"]["UA"].sum()

In [ ]:
nw.iterinfo = False

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(16, 8))

cycles = [
    ["LT-preheater", "LT-evaporator", "LT-condenser"],
    ["LHT-preheater", "HHT-preheater", "HT-evaporator", "HT-condenser"],
]

for row, cycle in enumerate(cycles):
    for col, heatex in enumerate(cycle):
        comp = nw.get_comp(heatex)
        heat = comp.Q_sections.val
        Th = comp.T_hot_sections.val
        Tc = comp.T_cold_sections.val
        fluidh = list(comp.inl[0].fluid.val.keys())[-1]
        fluidc = list(comp.inl[1].fluid.val.keys())[-1]
        axs[row, col].plot(heat, Th, "r-o", ms=3, label=fluidh)
        axs[row, col].plot(heat, Tc, "b-o", ms=3, label=fluidc)
        axs[row, col].set_title(f"{heatex} (min approach = {(Th - Tc).min():.2f} K)")

        axs[row, col].set_xlabel("cumulative heat duty Q [kW]")
        axs[row, col].set_ylabel("temperature T [degC]")
        axs[row, col].grid(alpha=0.3)
        axs[row, col].legend()

axs[0, 3].axis("off")  # LT row has only three heat exchangers

fig.tight_layout()
plt.show()

### Validation of the design point against Table 3 (Heberle et al., 2015)

In [ ]:
import pandas as pd


ht_evaporator.set_attr(td_pinch=ht_evaporator.td_pinch.val)
gC.set_attr(T=None)
nw.solve("design")
E_GF_PAPER = 5309.64 / 0.5222  # kW, brine exergy flow implied by Table 3 (StanMix)

# Table 3, simulation StanMix column (* = set variable in this model too).
table3 = {
    # HT-ORC
    "T_01 [degC]":         22.81,
    "T_02 [degC]":         24.01,
    "p_02 [bar]":          15.24,
    "T_03 [degC]":         60.23,
    "T_04 [degC]":        103.16,
    "T_05 [degC]":        104.23,
    "T_05a [degC]":        47.70,
    "m_HT_wf [kg/s]":     109.61,
    # LT-ORC
    "T_07 [degC]":         23.03,
    "T_08 [degC]":         23.51,
    "T_09 [degC]":         67.37,
    "T_10 [degC]":         69.37,
    "p_10 [bar]":           5.97,
    "T_10a [degC]":        40.02,
    "m_LT_wf [kg/s]":      69.72,
    # geothermal fluid
    "T_B [degC]":         106.96,
    "T_C [degC] *":        92.95,
    "T_D [degC] *":        70.37,
    "m_E [kg/s]":          54.48,
    "T_F [degC]":          52.13,
    "T_G [degC] *":        51.66,
    "T_H [degC]":          51.87,
    # power and efficiency
    "gross power [kW]":  5309.64,
    "HT feed pump [kW]":  226.21,
    "LT feed pump [kW]":   59.76,
    "eta_II_gross [%]":    52.22,
    "eta_II_net [%]":      49.41,
}

tespy_design = {
    "T_01 [degC]":       nw.get_conn("01").T.val,
    "T_02 [degC]":       nw.get_conn("02").T.val,
    "p_02 [bar]":        nw.get_conn("02").p.val,
    "T_03 [degC]":       nw.get_conn("03").T.val,
    "T_04 [degC]":       nw.get_conn("04").T.val,
    "T_05 [degC]":       nw.get_conn("05").T.val,
    "T_05a [degC]":      nw.get_conn("05a").T.val,
    "m_HT_wf [kg/s]":    nw.get_conn("01").m.val,
    "T_07 [degC]":       nw.get_conn("07").T.val,
    "T_08 [degC]":       nw.get_conn("08").T.val,
    "T_09 [degC]":       nw.get_conn("09").T.val,
    "T_10 [degC]":       nw.get_conn("10").T.val,
    "p_10 [bar]":        nw.get_conn("10").p.val,
    "T_10a [degC]":      nw.get_conn("10a").T.val,
    "m_LT_wf [kg/s]":    nw.get_conn("07").m.val,
    "T_B [degC]":        nw.get_conn("B").T.val,
    "T_C [degC] *":      nw.get_conn("C").T.val,
    "T_D [degC] *":      nw.get_conn("D1").T.val,
    "m_E [kg/s]":        nw.get_conn("E").m.val,
    "T_F [degC]":        nw.get_conn("F").T.val,
    "T_G [degC] *":      nw.get_conn("G").T.val,
    "T_H [degC]":        nw.get_conn("H").T.val,
    "gross power [kW]":  gross_power(),
    "HT feed pump [kW]": nw.get_conn("HT-e4").E.val,
    "LT feed pump [kW]": nw.get_conn("LT-e4").E.val,
    # efficiencies expressed in the paper's implied exergy reference (see markdown above)
    "eta_II_gross [%]":  100 * gross_power() / E_GF_PAPER,
    "eta_II_net [%]":    100 * net_power() / E_GF_PAPER,
}

validation = pd.DataFrame({"paper (Table 3)": table3, "TESPy": tespy_design})
validation["deviation"] = validation["TESPy"] - validation["paper (Table 3)"]
validation.round(2)

## Comparison with Figure 6 (Heberle et al., 2015)

In [ ]:
import pandas as pd

paper_fig6 = pd.DataFrame({
    "mass_flow_lht": [30,    35,    40,    45,    50,    55,    60,    65,    70,    75,    78,    80,    85,    90   ],
    "eta_II_gross":  [48.81, 49.30, 49.78, 50.25, 50.73, 51.21, 51.69, 52.17, 52.65, 53.13, 53.43, 53.50, 53.48, 53.48],
    "T_C":           [87.32, 88.05, 88.83, 89.65, 90.47, 91.25, 92.07, 92.89, 93.63, 94.41, 94.82, 95.19, 95.10, 95.10],
    "T_F":           [62.46, 61.64, 60.70, 59.59, 58.32, 56.89, 55.25, 53.28, 50.98, 48.20, 46.27, 44.75, 41.31, 36.80],
    "m_LT_wf":       [52.09, 54.30, 56.72, 59.35, 61.80, 64.30, 66.76, 69.26, 71.56, 74.06, 75.29, 76.35, 76.35, 76.35],
}).set_index("mass_flow_lht")

In [ ]:
# breakpoint solve: both specs active, LHT mass flow free
gG.set_attr(T=51.66)
gD2.set_attr(m=None)
lht_preheater.set_attr(td_pinch=3)
nw.solve("design")
m_star = gD2.m.val
print(f"regime switch (T_G = 51.66 degC and pinch = 3 K): m_LHT* = {m_star:.2f} kg/s")


def collect_kpis():
    return {
        "eta_II_gross": eta_II_gross(),
        "eta_II_net": eta_II_net(),
        "T_C": gC.T.val,
        "T_F": gF.T.val,
        "T_G": gG.T.val,
        "T_reinjection": gH.T.val,
        "m_LT_wf": c7.m.val,
        "lht_pinch": lht_preheater.td_pinch.val,
        "gross_power": gross_power(),
        "power_generation": net_power(),
        "status": nw.status,
    }


def solve_point(m):
    gD2.set_attr(m=m)
    if m < m_star:
        lht_preheater.set_attr(td_pinch=None)
        gG.set_attr(T=51.66)
        regime = "T_G fixed"
    else:
        gG.set_attr(T=None)
        lht_preheater.set_attr(td_pinch=3)
        regime = "pinch limited"
    nw.solve("design")
    res = collect_kpis()
    res["regime"] = regime
    return res

# same grid as the digitized Figure 6 data (5 kg/s steps + extra point at 78)
sweep = {}
for m in [30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 78, 80, 85, 90]:
    try:
        sweep[m] = solve_point(m)
    except Exception as err:
        sweep[m] = {"status": f"failed: {err}"}

tespy_fig6 = pd.DataFrame(sweep).T.sort_index()
tespy_fig6.index.name = "mass_flow_lht"

In [ ]:
# difference table: TESPy model vs. paper (positive = model higher).
num_cols = ["T_C", "T_F", "m_LT_wf", "gross_power"]
ok = tespy_fig6[pd.to_numeric(tespy_fig6["status"], errors="coerce") == 0]
common = paper_fig6.index.intersection(ok.index)

paper = paper_fig6.loc[common, ["eta_II_gross", "T_C", "T_F", "m_LT_wf"]].copy()
paper["gross_power"] = paper["eta_II_gross"] / 100 * E_GF_PAPER

tespy = ok.loc[common, num_cols].astype(float)
tespy["eta_II_gross"] = 100 * tespy["gross_power"] / E_GF_PAPER

cols = ["gross_power", "eta_II_gross", "T_C", "T_F", "m_LT_wf"]
comparison = pd.concat({"paper": paper[cols], "tespy": tespy[cols]}, axis=1)
for c in cols:
    comparison[("delta", c)] = comparison[("tespy", c)] - comparison[("paper", c)]

comparison = comparison.sort_index(axis=1, level=0, sort_remaining=False)
comparison.round(2)

In [ ]:
plot_cols = ["T_C", "T_F", "m_LT_wf", "T_reinjection", "gross_power"]
okp = tespy_fig6[pd.to_numeric(tespy_fig6["status"], errors="coerce") == 0][plot_cols].astype(float)
okp["eta_paper_ref"] = 100 * okp["gross_power"] / E_GF_PAPER

fig, ax1 = plt.subplots(figsize=(8.5, 5.5))
ax2 = ax1.twinx()

ax1.plot(paper_fig6.index, paper_fig6["eta_II_gross"], "ks--", mfc="none", label="$\\eta_{II,gross}$ paper")
ax1.plot(okp.index, okp["eta_paper_ref"], "ks-", label="$\\eta_{II,gross}$ TESPy")

for col, color, name in [("T_C", "tab:red", "$T_{GF,C}$"), ("T_F", "tab:blue", "$T_{GF,F}$"),
                         ("m_LT_wf", "tab:green", "$\\dot m_{LT}$")]:
    ax2.plot(paper_fig6.index, paper_fig6[col], "o--", color=color, mfc="none", label=f"{name} paper")
    ax2.plot(okp.index, okp[col], "o-", color=color, label=f"{name} TESPy")

ax1.set_xlabel("LHT mass flow rate [kg/s]")
ax1.set_ylabel("gross second law efficiency [%]")
ax2.set_ylabel("temperature [°C] / LT mass flow rate [kg/s]")
ax1.set_ylim(46, 60)
ax2.set_ylim(30, 100)
ax1.grid(alpha=0.3)
ax1.legend(loc="upper left", fontsize=8)
ax2.legend(loc="lower right", fontsize=8)
ax1.set_title("Reproduction of Heberle et al. (2015), Figure 6")
fig.tight_layout()
plt.show()